In [1]:
using LowLevelFEM, LinearAlgebra

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (caches not reused: 3 for different dependency version already loaded, 4 for file size changed)
Precompiling packages...
   0.2 s  ✓ LibCURL_jll
  38.5 s  ✓ LowLevelFEM
  1 dependency successfully precompiled in 42 seconds. 134 already precompiled.


In [2]:
openGeometry("boxes.geo")

In [3]:
#openPreProcessor()

In [4]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [5]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 2000)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

@time u = solveField(Symmetric(K), f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=false)

 15.331926 seconds (13.15 M allocations: 755.111 MiB, 2.51% gc time, 95.47% compilation time)


0

In [6]:
contact_pair = contact(u, master="master", slave="slave", cn=1e8)

Contact("slave" -> "master", 788 candidate nodes, 402 active, G=(2364, 9438), C=(2364, 2364))

In [7]:
support = [bc_bottom, bc_top]
free = freeDoFs(U, support)

u_it = copy(u)

old_tags = copy(contact_pair.master_element_tags)
old_G = copy(contact_pair.G)

for iter in 1:40

    updateContact!(contact_pair, u_it)

    (; G, C, g) = contact_pair

    nchanged = count(old_tags .!= contact_pair.master_element_tags)

    dG = norm(G - old_G) /
         max(norm(old_G), eps())

    println(
        "master changes = ", nchanged,
        ", dG = ", dG
    )

    old_tags = copy(contact_pair.master_element_tags)
    old_G = copy(G)

    # Penalty contact
    p  = -C * g
    rc = -G' * p
    Kc =  G' * C * G

    # Equilibrium residual and tangent
    r = K * u_it - f + rc
    A = K + Kc

    # Homogeneous Newton correction on prescribed DoFs
    Δu = vectorField(U, "body", [0, 0, 0])
    DoFs(Δu)[free] = -A[free, free] \ DoFs(r)[free]

    r0 = norm(DoFs(r)[free])

    α = 1.0
    u_trial = copy(u_it)
    r_trial = nothing

    while α > 1e-6

        u_trial = u_it + α * Δu

        updateContact!(contact_pair, u_trial)

        (; G, C, g) = contact_pair

        p_trial = -C * g
        rc_trial = -G' * p_trial

        r_trial = K * u_trial - f + rc_trial

        if norm(DoFs(r_trial)[free]) < r0
            break
        end

        α *= 0.5
    end

    u_it = copy(u_trial)
    r = r_trial

    err = α * norm(DoFs(Δu)[free]) /
          max(norm(DoFs(u_it)), eps())

    println(
        "iter = ", iter,
        ", α = ", α,
        ", active = ", count(contact_pair.active),
        ", min gap = ", minimum(contact_pair.gap_values),
        ", error = ", err,
        ", |r| = ", norm(DoFs(r)[free])
    )

    err < 1e-8 && break
end

u = u_it

master changes = 0, dG = 2.6072244790976894e-16
iter = 1, α = 1.0, active = 217, min gap = -5.1913221082751514e-5, error = 0.292307981479865, |r| = 27122.149385269757
master changes = 2, dG = 0.06560962236338808
iter = 2, α = 0.0009765625, active = 301, min gap = -5.18710104752678e-5, error = 4.178513745506329e-5, |r| = 26812.21153596191
master changes = 0, dG = 0.00017700041760377135
iter = 3, α = 0.001953125, active = 330, min gap = -5.1786530248405536e-5, error = 3.0154250164899304e-5, |r| = 26751.95295877488
master changes = 0, dG = 3.475648484722085e-5
iter = 4, α = 0.001953125, active = 337, min gap = -5.1702212257013846e-5, error = 2.7347680770680816e-5, |r| = 26713.023915557267
master changes = 0, dG = 1.536249476873863e-5
iter = 5, α = 0.00390625, active = 342, min gap = -5.153387105832684e-5, error = 5.433422533001548e-5, |r| = 26624.992015249816
master changes = 0, dG = 2.7544947916713794e-5
iter = 6, α = 0.00390625, active = 347, min gap = -5.136618634590297e-5, error = 5.4

nodal VectorField
[0.0; 0.0; … ; -0.20132082419205072; 0.023823146615548862;;]

In [8]:
showDoFResults(u, name="u cont.", visible=true, factor=1)

1

In [9]:
showElementResults(contact_pair.gap, name="gap")

2

In [10]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()


Két vagy több párnál majd:

```Julia
contacts = ContactSet(c1, c2, c3)

updateContact!(contacts, u_it)

Kc = sum(c.G' * c.C * c.G for c in contacts)
rc = sum(c.G' * c.C * c.g for c in contacts)

r = K * u_it - f + rc
A = K + Kc
```